<a href="https://colab.research.google.com/github/waghmodedevidas121-cloud/PARAM/blob/main/OmniAvatar_1_3B_T4_Colab_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🧑‍🎤 OmniAvatar 1.3B — T4 Low-VRAM Gradio Studio

**Photo + speech + behavior prompt → newly generated face and adaptive body animation.**

This is a separate, genuinely generative backend—not Ditto face-only animation and not Video Twin motion replay.

- **Model:** [OmniAvatar 1.3B](https://github.com/Omni-Avatar/OmniAvatar)
- **Inputs:** one image, speech audio, detailed motion/scene prompt
- **Output:** 480p-class generated video with lip, expression, head, shoulder, hand, and body motion
- **Target:** NVIDIA T4 15 GB with aggressive official CPU/VRAM offloading
- **License:** Apache‑2.0
- **API keys:** none

> Use only images and voices you own or have explicit permission to animate. Disclose synthetic media. Do not use this notebook for impersonation, fraud, harassment, or non-consensual content.


## Read this before running

OmniAvatar 1.3B is more advanced than Ditto, but **much slower**:

- Download: approximately **18–19 GB** of pinned model files.
- Free disk recommended: **30 GB+**.
- System RAM: **16 GB minimum; 20–24 GB recommended**. A low-RAM runtime may be killed while loading the 11.4 GB T5 encoder.
- T4 does not support hardware BF16, so this notebook uses **FP16 + SDPA** and deliberately skips FlashAttention.
- Begin with **2–4 seconds of audio**, 10–20 steps, and the 12k/16k token profile.
- A short clip can take **tens of minutes or longer** on a T4. The official 14B quality is substantially higher, but it is not a T4 model.

Community testing reported that the 1.3B model can run with 8 GB VRAM + 16 GB RAM, but a 6-second/50-step test took over 35 minutes. Results and time vary by runtime. [Reference](https://github.com/Omni-Avatar/OmniAvatar/issues/19).


## 0 · GPU, RAM, and disk check

Select **Runtime → Change runtime type → T4 GPU**. Enable **High RAM** too if your Colab plan exposes it.


In [ ]:
import os, shutil, subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    text=True, capture_output=True,
)
if gpu.returncode:
    raise RuntimeError("No NVIDIA GPU. Select Runtime → Change runtime type → T4 GPU.")
print("GPU:", gpu.stdout.strip())

mem_kb = 0
with open("/proc/meminfo") as handle:
    for row in handle:
        if row.startswith("MemTotal:"):
            mem_kb = int(row.split()[1])
            break
ram_gib = mem_kb / 1024**2
free_gib = shutil.disk_usage("/content").free / 1024**3
print(f"System RAM: {ram_gib:.1f} GiB")
print(f"Free disk:  {free_gib:.1f} GiB")

if free_gib < 30:
    raise RuntimeError("At least 30 GiB free disk is required. Reset/delete the runtime first.")
if ram_gib < 15:
    print("\n⚠️ LOW-RAM WARNING: model loading may be killed. Use a High-RAM runtime if possible.")
else:
    print("\n✅ Hardware pre-check passed.")


## 1 · Download pinned OmniAvatar source


In [ ]:
import shutil, subprocess
from pathlib import Path

ROOT = Path("/content/OmniAvatar")
COMMIT = "1536bf31abaec74364fb7d5883470d5b23ffa7f8"
if ROOT.exists() and not (ROOT / ".git").exists():
    shutil.rmtree(ROOT)
if not (ROOT / ".git").exists():
    subprocess.run([
        "git", "clone", "--filter=blob:none", "--no-checkout",
        "https://github.com/Omni-Avatar/OmniAvatar.git", str(ROOT),
    ], check=True)
subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", COMMIT], check=True)
print("OmniAvatar source:", subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True
).strip())


## 2 · Install isolated Python 3.10 environment

FlashAttention is intentionally omitted because it does not support the T4/Turing architecture. PyTorch SDPA is used instead.


In [ ]:
import os, subprocess, sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"
ENV = Path("/content/omniavatar-env")
PYTHON = ENV / "bin/python"

def shell(command):
    print("+", command)
    subprocess.run(["bash", "-lc", command], check=True)

shell("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1 libgl1 build-essential python3-dev fonts-dejavu-core")
shell(f"{sys.executable} -m pip install -q 'uv>=0.8,<1'")
shell(f"{sys.executable} -m uv venv --python 3.10 --clear {ENV}")
shell(
    f"{sys.executable} -m uv pip install --python {PYTHON} "
    "torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 "
    "--index-url https://download.pytorch.org/whl/cu121"
)

packages = [
    "numpy==1.26.4",
    "scipy==1.14.0",
    "librosa==0.10.2.post1",
    "soundfile==0.13.0",
    "peft==0.15.1",
    "transformers==4.52.3",
    "xfuser==0.4.1",
    "diffusers==0.33.1",
    "opencv-python==4.10.0.84",
    "ftfy",
    "einops",
    "tqdm==4.67.1",
    "gradio==5.44.1",
    "huggingface-hub==0.34.4",
    "hf-xet>=1.1.5,<2",
    "imageio==2.36.1",
    "imageio-ffmpeg==0.5.1",
    "Pillow==11.0.0",
    "PyYAML>=6,<7",
    "psutil",
    "setuptools==75.1.0",
]
quoted = " ".join(repr(item) for item in packages)
shell(f"{sys.executable} -m uv pip install --python {PYTHON} {quoted}")
print("Environment ready:", PYTHON)


## 3 · Apply T4 compatibility profile

Changes are limited to FP16, low-VRAM model wrapping, conservative generation defaults, and correcting OmniAvatar's hard-coded 14B TeaCache ID when the 1.3B checkpoint is used.


In [ ]:
from pathlib import Path

ROOT = Path("/content/OmniAvatar")
script_path = ROOT / "scripts" / "inference.py"
source = script_path.read_text(encoding="utf-8")
old = 'tea_cache_model_id="Wan2.1-T2V-14B")'
new = 'tea_cache_model_id="Wan2.1-T2V-1.3B" if "1.3B" in args.dit_path else "Wan2.1-T2V-14B")'
if old in source:
    script_path.write_text(source.replace(old, new), encoding="utf-8")
elif new not in source:
    raise RuntimeError("Could not apply the pinned TeaCache compatibility patch.")

# Wav2Vec is only needed once. Move its weights off GPU before diffusion.
audio_old = "            audio_prefix = torch.zeros_like(audio_embeddings[:first_fixed_frame])\n        else:\n            audio_embeddings = None"
audio_new = "            audio_prefix = torch.zeros_like(audio_embeddings[:first_fixed_frame])\n            del hidden_states, input_values\n            self.audio_encoder.to(\"cpu\")\n            torch.cuda.empty_cache()\n        else:\n            audio_embeddings = None"
source = script_path.read_text(encoding="utf-8")
if audio_old in source:
    script_path.write_text(source.replace(audio_old, audio_new), encoding="utf-8")
elif audio_new not in source:
    raise RuntimeError("Could not apply the Wav2Vec offload patch.")

config = r"""# T4 / Turing compatibility profile
dtype: "fp16"
text_encoder_path: pretrained_models/Wan2.1-T2V-1.3B/models_t5_umt5-xxl-enc-bf16.pth
image_encoder_path: None
dit_path: pretrained_models/Wan2.1-T2V-1.3B/diffusion_pytorch_model.safetensors
vae_path: pretrained_models/Wan2.1-T2V-1.3B/Wan2.1_VAE.pth
wav2vec_path: pretrained_models/wav2vec2-base-960h
exp_path: pretrained_models/OmniAvatar-1.3B
num_persistent_param_in_dit: 7000000000
reload_cfg: true
sp_size: 1
seed: 42
image_sizes_720: [[400, 720], [720, 720], [720, 400]]
image_sizes_1280: [[720, 720], [528, 960], [960, 528], [720, 1280], [1280, 720]]
max_hw: 720
max_tokens: 16000
seq_len: 200
overlap_frame: 13
guidance_scale: 4.5
audio_scale: 5.0
num_steps: 20
fps: 25
sample_rate: 16000
negative_prompt: "Vivid color tones, background or camera moving quickly, screen switching, subtitles, special effects, mutation, overexposed, static, blurred details, painting, still image, worst quality, low quality, JPEG artifacts, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn face, deformed, disfigured, malformed limbs, fused fingers, motionless image, chaotic or crowded background, extra limbs, walking backward"
silence_duration_s: 0.3
use_fsdp: false
tea_cache_l1_thresh: 0.0
"""
config_path = ROOT / "configs" / "inference_t4.yaml"
config_path.write_text(config, encoding="utf-8")
print("Wrote", config_path)


## 4 · Download pinned model files

This downloads the official Wan 2.1 1.3B base, OmniAvatar adapter, and Wav2Vec audio encoder. The `.pth`/`.pt` files are pinned to official repositories but use PyTorch serialization; only run checkpoints you trust.


In [ ]:
import subprocess, textwrap

PYTHON = "/content/omniavatar-env/bin/python"
download = r"""
import os
from pathlib import Path
from huggingface_hub import snapshot_download

os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
root = Path('/content/OmniAvatar/pretrained_models')

snapshot_download(
    repo_id='Wan-AI/Wan2.1-T2V-1.3B',
    revision='37ec512624d61f7aa208f7ea8140a131f93afc9a',
    local_dir=root / 'Wan2.1-T2V-1.3B',
    allow_patterns=[
        'config.json',
        'Wan2.1_VAE.pth',
        'diffusion_pytorch_model.safetensors',
        'models_t5_umt5-xxl-enc-bf16.pth',
        'google/umt5-xxl/*',
    ],
    max_workers=2,
)
snapshot_download(
    repo_id='OmniAvatar/OmniAvatar-1.3B',
    revision='5e1fb8803d9200848fe021952e0fd8f11e997bf6',
    local_dir=root / 'OmniAvatar-1.3B',
    allow_patterns=['config.json', 'pytorch_model.pt'],
    max_workers=2,
)
snapshot_download(
    repo_id='facebook/wav2vec2-base-960h',
    revision='22aad52d435eb6dbaf354bdad9b0da84ce7d6156',
    local_dir=root / 'wav2vec2-base-960h',
    allow_patterns=[
        'config.json', 'feature_extractor_config.json', 'preprocessor_config.json',
        'model.safetensors', 'special_tokens_map.json', 'tokenizer_config.json', 'vocab.json',
    ],
    max_workers=2,
)

required = [
    root / 'Wan2.1-T2V-1.3B/diffusion_pytorch_model.safetensors',
    root / 'Wan2.1-T2V-1.3B/models_t5_umt5-xxl-enc-bf16.pth',
    root / 'Wan2.1-T2V-1.3B/Wan2.1_VAE.pth',
    root / 'Wan2.1-T2V-1.3B/google/umt5-xxl/spiece.model',
    root / 'OmniAvatar-1.3B/pytorch_model.pt',
    root / 'wav2vec2-base-960h/model.safetensors',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError('Missing model files: ' + ', '.join(missing))
total = sum(path.stat().st_size for path in root.rglob('*') if path.is_file())
print(f'Model download complete: {total / 1024**3:.2f} GiB')
"""
subprocess.run([PYTHON, "-c", textwrap.dedent(download)], check=True)


## 5 · Verify environment (does not load the 18 GB model)


In [ ]:
import os, subprocess, textwrap

PYTHON = "/content/omniavatar-env/bin/python"
check = r"""
import torch, torchvision, transformers, peft, diffusers, gradio, librosa, xfuser
print('torch:', torch.__version__, '| CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('transformers:', transformers.__version__, '| peft:', peft.__version__, '| diffusers:', diffusers.__version__)
print('gradio:', gradio.__version__, '| librosa:', librosa.__version__)
print('xfuser:', getattr(xfuser, '__version__', 'import OK'))
assert torch.cuda.is_available(), 'The isolated environment cannot see the GPU.'
"""
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
result = subprocess.run([PYTHON, '-c', textwrap.dedent(check)], text=True, capture_output=True, env=env)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode:
    raise RuntimeError('Environment verification failed; read the detailed output above.')


## 6 · Create the OmniAvatar Gradio app

The Gradio server stays lightweight. Each queued job starts the official `torchrun` inference in an isolated child process, streams its log to this notebook, then releases all model RAM/VRAM when finished.


In [ ]:
from pathlib import Path

APP_PATH = Path("/content/OmniAvatar/omniavatar_gradio.py")
APP_CODE = 'from __future__ import annotations\n\nimport argparse\nimport gc\nimport os\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport traceback\nimport uuid\nfrom pathlib import Path\n\n# Colab exports an inline Matplotlib backend that is unavailable in the venv.\nos.environ["MPLBACKEND"] = "Agg"\nos.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\nprint("[startup 1/3] Importing Gradio and media libraries…", flush=True)\nimport gradio as gr\nimport soundfile as sf\nimport yaml\nfrom PIL import Image, ImageOps\n\nROOT = Path(__file__).resolve().parent\nCONFIG_PATH = ROOT / "configs" / "inference_t4.yaml"\nOUTPUT_ROOT = ROOT / "outputs" / "omniavatar_gradio"\nDEMO_ROOT = ROOT / "demo_out"\nOUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\nos.chdir(ROOT)\n\nFFMPEG_BIN = shutil.which("ffmpeg")\nif FFMPEG_BIN is None:\n    from imageio_ffmpeg import get_ffmpeg_exe\n\n    bundled_ffmpeg = Path(get_ffmpeg_exe()).resolve()\n    shim_dir = OUTPUT_ROOT / ".bin"\n    shim_dir.mkdir(parents=True, exist_ok=True)\n    shim = shim_dir / "ffmpeg"\n    if not shim.exists():\n        shim.symlink_to(bundled_ffmpeg)\n    os.environ["PATH"] = f"{shim_dir}:{os.environ.get(\'PATH\', \'\')}"\n    FFMPEG_BIN = str(shim)\n\nTORCHRUN = Path(sys.executable).parent / "torchrun"\nMAX_AUDIO_SECONDS = 8.0\nMODEL_LOCK = threading.Lock()\n\nDEFAULT_NEGATIVE = (\n    "Vivid color tones, background or camera moving quickly, screen switching, subtitles, "\n    "special effects, mutation, overexposed, static, blurred details, painting, still image, "\n    "worst quality, low quality, JPEG artifacts, ugly, incomplete, extra fingers, poorly drawn "\n    "hands, poorly drawn face, deformed, disfigured, malformed limbs, fused fingers, motionless "\n    "image, chaotic or crowded background, extra limbs, walking backward"\n)\n\nPROMPT_PRESETS = {\n    "Natural presenter": (\n        "A realistic waist-up video of a confident presenter speaking directly to the camera. "\n        "The presenter uses natural, dynamic, rhythmic hand gestures that complement the speech. "\n        "Both hands remain clearly visible and unobstructed. Facial expressions are expressive "\n        "and emotionally appropriate. The camera is locked and steady, with soft studio lighting, "\n        "sharp details, realistic skin, and a clean professional background."\n    ),\n    "Calm instructor": (\n        "A realistic waist-up instructor speaking calmly to the camera, using subtle open-palm "\n        "hand gestures and occasional natural nods. The delivery is warm and trustworthy. Hands "\n        "stay visible below the face. Locked camera, even studio light, clean background, sharp detail."\n    ),\n    "Energetic creator": (\n        "A realistic social-media presenter speaking energetically to the camera with lively but "\n        "controlled hand gestures, expressive eyebrows, smiles, and natural upper-body movement. "\n        "Hands are fully visible and never cover the face. Locked camera, bright clean studio, sharp detail."\n    ),\n    "Storyteller": (\n        "A realistic waist-up storyteller speaking directly to the camera with emotionally varied "\n        "expressions, gentle head turns, and meaningful hand gestures timed to the delivery. The "\n        "camera remains stationary. Cinematic soft lighting, stable clean background, realistic detail."\n    ),\n}\n\n\ndef clean_old_jobs(max_age_hours: float = 10.0) -> None:\n    cutoff = time.time() - max_age_hours * 3600\n    for item in OUTPUT_ROOT.iterdir():\n        try:\n            if item.name.startswith("."):\n                continue\n            if item.is_dir() and item.stat().st_mtime < cutoff:\n                shutil.rmtree(item, ignore_errors=True)\n        except OSError:\n            pass\n\n\ndef run_checked(command: list[str]) -> None:\n    result = subprocess.run(command, text=True, capture_output=True)\n    if result.returncode:\n        detail = (result.stderr or result.stdout)[-3000:]\n        raise RuntimeError(f"Command failed ({result.returncode}):\\n{detail}")\n\n\ndef prepare_image(source_path: str, destination: Path) -> None:\n    Image.MAX_IMAGE_PIXELS = 40_000_000\n    with Image.open(source_path) as opened:\n        image = ImageOps.exif_transpose(opened).convert("RGB")\n        width, height = image.size\n        if min(width, height) < 384:\n            raise ValueError("Image is too small. Use at least 384 px on each side.")\n        if width * height > 40_000_000:\n            raise ValueError("Image is too large. Use an image below 40 megapixels.")\n        image.save(destination, format="JPEG", quality=95, subsampling=0)\n\n\ndef prepare_audio(source_path: str, destination: Path) -> float:\n    run_checked([\n        FFMPEG_BIN,\n        "-hide_banner",\n        "-loglevel",\n        "error",\n        "-y",\n        "-i",\n        source_path,\n        "-vn",\n        "-ac",\n        "1",\n        "-ar",\n        "16000",\n        "-c:a",\n        "pcm_s16le",\n        str(destination),\n    ])\n    audio_info = sf.info(str(destination))\n    duration = float(audio_info.frames / audio_info.samplerate)\n    if duration < 0.5:\n        raise ValueError("Audio is too short. Use at least 0.5 seconds.")\n    if duration > MAX_AUDIO_SECONDS:\n        raise ValueError(\n            f"Audio is {duration:.1f}s. This T4 notebook limits a job to {MAX_AUDIO_SECONDS:.0f}s "\n            "because generation is extremely slow. Trim the audio and combine clips later."\n        )\n    return duration\n\n\ndef add_watermark(source: Path, destination: Path) -> bool:\n    font = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"\n    vf = (\n        "drawbox=x=w-238:y=h-52:w=226:h=40:color=black@0.55:t=fill,"\n        f"drawtext=fontfile={font}:text=\'AI-generated avatar\':"\n        "fontcolor=white:fontsize=18:x=w-tw-20:y=h-th-20"\n    )\n    result = subprocess.run([\n        FFMPEG_BIN,\n        "-hide_banner",\n        "-loglevel",\n        "error",\n        "-y",\n        "-i",\n        str(source),\n        "-vf",\n        vf,\n        "-c:v",\n        "libx264",\n        "-preset",\n        "veryfast",\n        "-crf",\n        "18",\n        "-c:a",\n        "copy",\n        "-movflags",\n        "+faststart",\n        str(destination),\n    ], text=True, capture_output=True)\n    if result.returncode:\n        print("Watermark step skipped:", result.stderr[-1200:], flush=True)\n        return False\n    return True\n\n\ndef newest_result(previous: set[Path], started: float) -> Path | None:\n    candidates = list(DEMO_ROOT.rglob("result_000_000_wav.mp4")) if DEMO_ROOT.exists() else []\n    candidates = [\n        path for path in candidates\n        if path not in previous and path.stat().st_mtime >= started - 3\n    ]\n    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None\n\n\ndef tail_text(path: Path, max_chars: int = 6000) -> str:\n    try:\n        return path.read_text(encoding="utf-8", errors="replace")[-max_chars:]\n    except OSError:\n        return "No generation log was available."\n\n\ndef generate_avatar(\n    image_path: str,\n    audio_path: str,\n    prompt: str,\n    negative_prompt: str,\n    steps: int,\n    prompt_cfg: float,\n    audio_cfg: float,\n    max_tokens: int,\n    tea_cache: float,\n    seed: int,\n    disclosure_watermark: bool,\n    consent_confirmed: bool,\n    progress=gr.Progress(),\n):\n    if not consent_confirmed:\n        raise gr.Error("Confirm that you have permission to use the image and audio.")\n    if not image_path:\n        raise gr.Error("Upload one clear waist-up or full-body image.")\n    if not audio_path:\n        raise gr.Error("Upload or record speech audio.")\n    prompt = " ".join((prompt or "").replace("@@", " ").split())\n    negative_prompt = " ".join((negative_prompt or DEFAULT_NEGATIVE).replace("@@", " ").split())\n    if len(prompt) < 20:\n        raise gr.Error("Add a descriptive motion prompt of at least 20 characters.")\n\n    with MODEL_LOCK:\n        started_clock = time.perf_counter()\n        started_epoch = time.time()\n        clean_old_jobs()\n        job_dir = OUTPUT_ROOT / f"job_{time.strftime(\'%Y%m%d_%H%M%S\')}_{uuid.uuid4().hex[:8]}"\n        job_dir.mkdir(parents=True, exist_ok=False)\n        input_image = job_dir / "source.jpg"\n        input_audio = job_dir / "speech.wav"\n        input_txt = job_dir / "input.txt"\n        job_config = job_dir / "inference.yaml"\n        log_path = job_dir / "generation.log"\n        raw_output = job_dir / "omniavatar_raw.mp4"\n        final_output = job_dir / "omniavatar.mp4"\n\n        process: subprocess.Popen | None = None\n        try:\n            progress(0.03, desc="Validating inputs…")\n            prepare_image(image_path, input_image)\n            duration = prepare_audio(audio_path, input_audio)\n            input_txt.write_text(\n                f"{prompt}@@{input_image}@@{input_audio}\\n", encoding="utf-8"\n            )\n\n            prior_results = set(DEMO_ROOT.rglob("result_000_000_wav.mp4")) if DEMO_ROOT.exists() else set()\n            with CONFIG_PATH.open("r", encoding="utf-8") as config_file:\n                config = yaml.safe_load(config_file)\n            config.update({\n                "seed": max(0, int(seed)),\n                "num_steps": int(steps),\n                "guidance_scale": float(prompt_cfg),\n                "audio_scale": float(audio_cfg),\n                "max_tokens": int(max_tokens),\n                "tea_cache_l1_thresh": float(tea_cache),\n                "num_persistent_param_in_dit": 7000000000,\n                "overlap_frame": 13,\n                "negative_prompt": negative_prompt,\n            })\n            job_config.write_text(\n                yaml.safe_dump(config, sort_keys=False, allow_unicode=True),\n                encoding="utf-8",\n            )\n\n            master_port = random.randint(29600, 29990)\n            command = [\n                str(TORCHRUN),\n                "--standalone",\n                "--nproc_per_node=1",\n                f"--master_port={master_port}",\n                "scripts/inference.py",\n                "--config",\n                str(job_config),\n                "--input_file",\n                str(input_txt),\n            ]\n            env = os.environ.copy()\n            env.update({\n                "CUDA_VISIBLE_DEVICES": "0",\n                "MPLBACKEND": "Agg",\n                "PYTHONUNBUFFERED": "1",\n                "TOKENIZERS_PARALLELISM": "false",\n                "OMP_NUM_THREADS": "2",\n            })\n\n            progress(0.08, desc="Loading ~18 GB of model weights…")\n            print("\\n[OmniAvatar job]", " ".join(command), flush=True)\n            with log_path.open("w", encoding="utf-8") as log_file:\n                process = subprocess.Popen(\n                    command,\n                    cwd=ROOT,\n                    env=env,\n                    stdout=subprocess.PIPE,\n                    stderr=subprocess.STDOUT,\n                    text=True,\n                    bufsize=1,\n                )\n                assert process.stdout is not None\n                for line in process.stdout:\n                    print(line, end="", flush=True)\n                    log_file.write(line)\n                    log_file.flush()\n                return_code = process.wait()\n\n            if return_code != 0:\n                detail = tail_text(log_path)\n                if return_code in (-9, 137) or "SIGKILL" in detail:\n                    raise RuntimeError(\n                        "The process was killed by host RAM pressure. Select a High-RAM runtime "\n                        "or use a runtime with at least 16–24 GB system RAM.\\n" + detail[-1800:]\n                    )\n                raise RuntimeError(f"OmniAvatar exited with code {return_code}.\\n{detail[-2400:]}")\n\n            progress(0.92, desc="Collecting and finalizing the MP4…")\n            result = newest_result(prior_results, started_epoch)\n            if result is None or result.stat().st_size == 0:\n                raise RuntimeError(\n                    "Inference completed but its result MP4 was not found.\\n" + tail_text(log_path)[-2400:]\n                )\n            shutil.copy2(result, raw_output)\n            watermarked = disclosure_watermark and add_watermark(raw_output, final_output)\n            selected_output = final_output if watermarked else raw_output\n\n            elapsed = time.perf_counter() - started_clock\n            status = (\n                "### ✅ OmniAvatar video ready\\n"\n                f"- Input audio: **{duration:.1f}s** · Total job time: **{elapsed / 60:.1f} min**\\n"\n                f"- Steps: **{int(steps)}** · Prompt CFG: **{float(prompt_cfg):.1f}** · "\n                f"Audio CFG: **{float(audio_cfg):.1f}**\\n"\n                f"- Token budget: **{int(max_tokens)}** · TeaCache: **{float(tea_cache):.2f}**"\n            )\n            if disclosure_watermark and not watermarked:\n                status += "\\n- ⚠️ Watermarking failed; the raw generated result is shown."\n\n            # Keep the user-facing copy and log, remove the duplicate dynamic CLI folder.\n            dynamic_dir = result.parent\n            if DEMO_ROOT in dynamic_dir.parents:\n                shutil.rmtree(dynamic_dir, ignore_errors=True)\n            gc.collect()\n            progress(1.0, desc="Done")\n            return str(selected_output), status, str(log_path)\n        except gr.Error:\n            raise\n        except Exception as exc:\n            traceback.print_exc()\n            if process is not None and process.poll() is None:\n                process.terminate()\n            raise gr.Error(f"Generation failed: {exc}") from exc\n\n\nCSS = """\n.gradio-container {max-width: 1220px !important;}\n.hero {text-align:center; padding: 12px 0 4px;}\n.hero h1 {font-size:2.25rem; margin-bottom:.25rem;}\n.notice {border-left:4px solid #7c3aed; padding:12px 15px; background:rgba(124,58,237,.09); border-radius:8px;}\n.warning {border-left:4px solid #f59e0b; padding:10px 14px; background:rgba(245,158,11,.09); border-radius:8px;}\n"""\n\nprint("[startup 2/3] Building OmniAvatar interface…", flush=True)\nwith gr.Blocks(\n    theme=gr.themes.Soft(primary_hue="violet", secondary_hue="indigo"),\n    css=CSS,\n    title="OmniAvatar 1.3B — T4 Studio",\n) as demo:\n    gr.HTML("""\n    <div class="hero">\n      <h1>🧑\u200d🎤 OmniAvatar 1.3B Studio</h1>\n      <p>Photo + speech + behavior prompt → generative face and adaptive body animation</p>\n    </div>\n    """)\n    gr.Markdown(\n        "<div class=\'notice\'><b>True generative mode:</b> unlike Video Twin replay, OmniAvatar "\n        "generates new face, head, shoulder, hand, and body motion guided by the audio and prompt.</div>"\n    )\n    gr.Markdown(\n        "<div class=\'warning\'><b>T4 reality:</b> this model is extremely slow. Start with 2–4 seconds "\n        "of audio, 10–20 steps, and the Safe token profile. A 6-second clip can take tens of minutes "\n        "or longer. At least 16 GB system RAM is strongly recommended.</div>"\n    )\n\n    with gr.Row(equal_height=False):\n        with gr.Column(scale=5):\n            source_image = gr.Image(\n                label="1 · Source image (waist-up/full-body recommended)",\n                type="filepath",\n                sources=["upload", "webcam"],\n                height=410,\n            )\n            driving_audio = gr.Audio(\n                label="2 · Speech audio (0.5–8 seconds; start with 2–4s)",\n                type="filepath",\n                sources=["upload", "microphone"],\n            )\n            preset = gr.Dropdown(\n                choices=list(PROMPT_PRESETS),\n                value="Natural presenter",\n                label="3 · Motion-prompt preset",\n            )\n            prompt = gr.Textbox(\n                label="Behavior and scene prompt",\n                value=PROMPT_PRESETS["Natural presenter"],\n                lines=5,\n            )\n            preset.change(\n                fn=lambda name: PROMPT_PRESETS.get(name, PROMPT_PRESETS["Natural presenter"]),\n                inputs=preset,\n                outputs=prompt,\n                queue=False,\n            )\n\n            with gr.Accordion("Advanced generation controls", open=False):\n                negative_prompt = gr.Textbox(\n                    label="Negative prompt",\n                    value=DEFAULT_NEGATIVE,\n                    lines=4,\n                )\n                steps = gr.Slider(\n                    10, 30, value=20, step=5,\n                    label="Sampling steps",\n                    info="10 is experimental/fast; official guidance is 20–50.",\n                )\n                prompt_cfg = gr.Slider(3.0, 6.0, value=4.5, step=0.1, label="Prompt CFG")\n                audio_cfg = gr.Slider(3.0, 7.0, value=5.0, step=0.1, label="Audio CFG")\n                max_tokens = gr.Radio(\n                    choices=[\n                        ("T4 Safe · 12k (slowest, lowest peak)", 12000),\n                        ("T4 Balanced · 16k", 16000),\n                        ("Faster · 24k (more VRAM)", 24000),\n                        ("Official · 30k (highest peak)", 30000),\n                    ],\n                    value=16000,\n                    label="Per-segment token budget",\n                )\n                tea_cache = gr.Slider(\n                    0.0, 0.10, value=0.0, step=0.01,\n                    label="TeaCache acceleration",\n                    info="0 gives best quality. Higher is faster but can visibly reduce motion quality.",\n                )\n                seed = gr.Number(value=42, precision=0, label="Seed")\n                disclosure_watermark = gr.Checkbox(\n                    value=True, label="Add ‘AI-generated avatar’ disclosure watermark"\n                )\n\n            consent = gr.Checkbox(\n                value=False,\n                label=(\n                    "I own or have explicit permission to use this face and audio, and I will "\n                    "disclose synthetic media where appropriate."\n                ),\n            )\n            generate_button = gr.Button("✨ Generate with OmniAvatar", variant="primary", size="lg")\n\n            example_image = ROOT / "examples" / "images" / "0000.jpeg"\n            example_audio = ROOT / "examples" / "audios" / "0000.MP3"\n            if example_image.exists() and example_audio.exists():\n                gr.Examples(\n                    examples=[[str(example_image), str(example_audio)]],\n                    inputs=[source_image, driving_audio],\n                    label="Official OmniAvatar sample inputs",\n                )\n\n        with gr.Column(scale=6):\n            output_video = gr.Video(label="Generated OmniAvatar video", height=560)\n            status = gr.Markdown("Use a short test clip first, then generate.")\n            generation_log = gr.File(label="Generation log")\n\n    generate_button.click(\n        fn=generate_avatar,\n        inputs=[\n            source_image,\n            driving_audio,\n            prompt,\n            negative_prompt,\n            steps,\n            prompt_cfg,\n            audio_cfg,\n            max_tokens,\n            tea_cache,\n            seed,\n            disclosure_watermark,\n            consent,\n        ],\n        outputs=[output_video, status, generation_log],\n        api_name=False,\n    )\n\n    gr.Markdown(\n        "---\\n**Input tips:** use one front-facing person with visible shoulders and hands, a simple "\n        "background, and clean speech. Prompt the desired behavior explicitly. Avoid crowded scenes, "\n        "occluded hands, extreme profiles, and long first tests. The 1.3B checkpoint is a lightweight "\n        "research model; its quality is below the 14B version. "\n        "[Official OmniAvatar](https://github.com/Omni-Avatar/OmniAvatar) · "\n        "[Apache‑2.0 license](https://github.com/Omni-Avatar/OmniAvatar/blob/main/LICENSE.txt)"\n    )\n\nprint("[startup 3/3] Interface ready; starting Gradio…", flush=True)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--host", default="0.0.0.0")\n    parser.add_argument("--port", type=int, default=7861)\n    parser.add_argument("--share", action="store_true")\n    args = parser.parse_args()\n    demo.queue(max_size=2, default_concurrency_limit=1).launch(\n        server_name=args.host,\n        server_port=args.port,\n        share=args.share,\n        show_error=True,\n        allowed_paths=[str(OUTPUT_ROOT), str(ROOT / "examples")],\n    )\n\n\nif __name__ == "__main__":\n    main()\n'
APP_PATH.write_text(APP_CODE, encoding="utf-8")
print(f"Wrote {APP_PATH} ({APP_PATH.stat().st_size:,} bytes)")


## 7 · Launch Gradio

Open the printed `gradio.live` URL and keep this cell running. The model is loaded only after you click Generate. Do not share the URL.


In [ ]:
import os, subprocess

# Stop only an older OmniAvatar UI, not the Ditto UI on port 7860.
subprocess.run(["pkill", "-f", "/content/OmniAvatar/omniavatar_gradio.py"], check=False)
command = [
    "/content/omniavatar-env/bin/python", "-u",
    "/content/OmniAvatar/omniavatar_gradio.py",
    "--host", "0.0.0.0", "--port", "7861", "--share",
]
env = os.environ.copy()
env.update({"MPLBACKEND": "Agg", "GRADIO_ANALYTICS_ENABLED": "False", "PYTHONUNBUFFERED": "1"})
print("Launching OmniAvatar Gradio…", flush=True)
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env,
)
try:
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
finally:
    code = process.wait()
    if code not in (0, -2, -15):
        raise RuntimeError(f"Gradio exited with code {code}")


## Best first test

1. Upload one clear **waist-up image** with visible hands and simple background.
2. Use **2–3 seconds** of clean speech.
3. Select **Natural presenter**.
4. Use **10 steps**, **T4 Safe 12k**, TeaCache **0**.
5. Confirm permission and generate.

If that succeeds, move to 20 steps / 16k. Higher TeaCache is faster but can reduce quality. If the process reports `SIGKILL` or code `-9/137`, the failure is system RAM—not VRAM; use High RAM. If CUDA OOM occurs, select 12k tokens and shorten audio.
